# Progres 3 — Pemodelan SARIMA
**Proyek:** Peramalan Realisasi Bea Perolehan Hak atas Tanah dan Bangunan (BPHTB) Kota Surabaya Menggunakan Metode SARIMA dan Random Forest

**Tujuan:**
1. Mencari kombinasi orde SARIMA terbaik pada data latih (log realisasi, d = 0, D = 1, s = 12).
2. Memeriksa diagnostik residual (autokorelasi, normalitas).
3. Meramalkan 12 bulan data uji beserta interval prediksi 80%.

**Input:** `bphtb_bulanan.csv`, `pembagian_data.json` (dari Progres 1 dan 2)
**Output:** `sarima_kandidat.csv`, `sarima_order.json`, `sarima_prediksi_uji.csv`

In [ ]:
# ===== Pengaturan file (VS Code / Jupyter lokal) =====
# Semua file input dan output berada di folder yang sama dengan notebook ini.
# Jalankan dulu progres sebelumnya agar file yang dibutuhkan sudah tersedia.
# Library yang diperlukan (cukup sekali, di terminal dengan venv aktif):
#   pip install pandas numpy matplotlib scipy scikit-learn statsmodels openpyxl ipykernel
import os

def pastikan_file(nama):
    """Hentikan notebook dengan pesan jelas jika file input belum ada."""
    if not os.path.exists(nama):
        raise FileNotFoundError(
            f"File '{nama}' tidak ditemukan di folder {os.getcwd()}. "
            "Pastikan file ada di folder yang sama dengan notebook, "
            "atau jalankan progres sebelumnya terlebih dahulu.")

def unduh(*nama_file):
    """Di VS Code file sudah tersimpan di folder kerja; cukup tampilkan daftarnya."""
    for f in nama_file:
        print("Tersimpan:", os.path.abspath(f))

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
plt.rcParams["figure.figsize"] = (11, 4)

## 3.1 Memuat data dan pembagian latih–uji
Konfigurasi transformasi dan diferensiasi diambil dari `pembagian_data.json` (hasil Progres 2): log, d = 0, D = 1, s = 12.

In [ ]:
pastikan_file("bphtb_bulanan.csv"); pastikan_file("pembagian_data.json")
y = pd.read_csv("bphtb_bulanan.csv", index_col="bulan", parse_dates=True)["realisasi_miliar"].asfreq("MS")
cfg = json.load(open("pembagian_data.json"))
train, test = y[y.index < cfg["awal_uji"]], y[y.index >= cfg["awal_uji"]]
ly_train = np.log(train)
d, D, s = cfg["d"], cfg["D"], cfg["s"]
BURN = d + D * s          # observasi awal yang "terpakai" oleh diferensiasi

print(f"Latih: {len(train)} bulan ({train.index[0]:%b %Y} – {train.index[-1]:%b %Y})")
print(f"Uji  : {len(test)} bulan ({test.index[0]:%b %Y} – {test.index[-1]:%b %Y})")
print("Observasi efektif setelah diferensiasi:", len(train) - BURN)

## 3.2 Pencarian orde (grid search)
Semua kombinasi p, q, P, Q ∈ {0, 1}, dengan dan tanpa konstanta (`trend`), dicocokkan pada log data latih. Untuk setiap kandidat dicatat AIC, AICc (koreksi sampel kecil, penting karena observasi efektif hanya sekitar 44), BIC, serta p-value uji Ljung-Box pada lag 12.

**Aturan pemilihan:** kandidat dengan AIC terkecil di antara yang (a) konvergen dan (b) residualnya lolos Ljung-Box (p > 0,05). Jika tidak ada yang lolos, dipakai AIC terkecil secara umum.

In [ ]:
def fit_sarima(seri, order, seasonal, trend):
    mod = SARIMAX(seri, order=order, seasonal_order=seasonal, trend=trend)
    return mod.fit(disp=False, maxiter=300)

def aicc(res, n_eff):
    k = len(res.params)
    return res.aic + 2 * k * (k + 1) / (n_eff - k - 1)

baris = []
for p, q, P, Q in itertools.product([0, 1], repeat=4):
    for trend in ["n", "c"]:
        try:
            res_ = fit_sarima(ly_train, (p, d, q), (P, D, Q, s), trend)
            resid_ = res_.resid.iloc[BURN:]
            lb_p = acorr_ljungbox(resid_, lags=[12], model_df=p + q + P + Q, return_df=True)["lb_pvalue"].iloc[0]
            baris.append(dict(p=p, q=q, P=P, Q=Q, trend=trend, k=len(res_.params),
                              AIC=res_.aic, AICc=aicc(res_, len(train) - BURN), BIC=res_.bic,
                              LB_p12=lb_p, konvergen=bool((res_.mle_retvals or {}).get("converged", True))))
        except Exception as e:
            print("Gagal dicocokkan:", (p, q, P, Q, trend), type(e).__name__)

kandidat = pd.DataFrame(baris).sort_values("AIC").reset_index(drop=True)
kandidat.to_csv("sarima_kandidat.csv", index=False)
print(len(kandidat), "kandidat berhasil dicocokkan")

In [ ]:
layak = kandidat[kandidat["konvergen"] & (kandidat["LB_p12"] > 0.05)]
if len(layak) == 0:
    print("Peringatan: tidak ada kandidat yang lolos Ljung-Box; dipakai AIC terkecil.")
terpilih = (layak if len(layak) else kandidat).iloc[0].to_dict()
print("Terpilih: SARIMA({p},{d},{q})({P},{D},{Q}){s}, trend='{t}'".format(
      p=int(terpilih["p"]), d=d, q=int(terpilih["q"]), P=int(terpilih["P"]), D=D, Q=int(terpilih["Q"]), s=s, t=terpilih["trend"]))
print("AIC terkecil mutlak:", kandidat.iloc[0][["p", "q", "P", "Q", "trend"]].tolist(),
      "| AICc terkecil:", kandidat.sort_values("AICc").iloc[0][["p", "q", "P", "Q", "trend"]].tolist())
kandidat.head(10).round(3)

## 3.3 Model terpilih

In [ ]:
ORDER = (int(terpilih["p"]), d, int(terpilih["q"]))
SEASONAL = (int(terpilih["P"]), D, int(terpilih["Q"]), s)
TREND = terpilih["trend"]
res = fit_sarima(ly_train, ORDER, SEASONAL, TREND)
print(res.summary())

## 3.4 Diagnostik residual
Residual yang baik berperilaku seperti derau putih: tidak berautokorelasi, rata-rata nol, dan mendekati normal. Residual pada `BURN` observasi pertama dibuang karena belum terdefinisi oleh diferensiasi.

In [ ]:
resid = res.resid.iloc[BURN:]
z = (resid - resid.mean()) / resid.std()

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
z.plot(ax=ax[0, 0], marker="o", ms=3); ax[0, 0].axhline(0, color="k", lw=.8)
ax[0, 0].set_title("Residual terstandar"); ax[0, 0].grid(alpha=.3)
ax[0, 1].hist(z, bins=12, density=True, alpha=.6, label="Residual")
xs = np.linspace(-3.5, 3.5, 200); ax[0, 1].plot(xs, stats.norm.pdf(xs), "r", label="Normal baku")
ax[0, 1].set_title("Histogram residual"); ax[0, 1].legend()
stats.probplot(z, dist="norm", plot=ax[1, 0]); ax[1, 0].set_title("Q-Q plot")
plot_acf(resid, lags=min(20, len(resid) // 2 - 1), ax=ax[1, 1]); ax[1, 1].set_title("ACF residual")
plt.tight_layout(); plt.show()

In [ ]:
lb = acorr_ljungbox(resid, lags=[6, 12, 18], model_df=ORDER[0] + ORDER[2] + SEASONAL[0] + SEASONAL[2], return_df=True)
jb = stats.jarque_bera(resid); sw = stats.shapiro(resid)
print("Ljung-Box (H0: residual tidak berautokorelasi)"); print(lb.round(4))
print(f"\nJarque-Bera : stat={jb.statistic:.3f}, p={jb.pvalue:.4f}")
print(f"Shapiro-Wilk: stat={sw.statistic:.3f}, p={sw.pvalue:.4f}")
print("Rata-rata residual:", round(resid.mean(), 4))
print("\nKesimpulan autokorelasi:", "tidak ada autokorelasi tersisa (p > 0,05 di semua lag)"
      if (lb["lb_pvalue"] > .05).all() else "masih ada autokorelasi pada sebagian lag, model perlu dicermati")
print("Kesimpulan normalitas  :", "residual konsisten dengan normal" if min(jb.pvalue, sw.pvalue) > .05
      else "residual menyimpang dari normal (kemungkinan akibat bulan-bulan lonjakan); interval prediksi perlu dibaca hati-hati")

## 3.5 Kecocokan pada data latih

In [ ]:
fit_in = np.exp(res.fittedvalues).iloc[BURN:]
fig, ax = plt.subplots()
train.plot(ax=ax, label="Aktual", marker="o", ms=3)
fit_in.plot(ax=ax, label="Fitted SARIMA", color="tab:orange")
ax.set_title("SARIMA pada data latih (miliar Rp)"); ax.legend(); ax.grid(alpha=.3); plt.show()

## 3.6 Peramalan data uji (12 bulan)
Ramalan dibuat pada skala log lalu dikembalikan dengan `exp`. Interval 80% dihitung pada skala log lalu ditransformasi, sehingga tidak simetris.

In [ ]:
fc = res.get_forecast(steps=len(test))
mu = np.asarray(fc.predicted_mean)
ci = np.asarray(fc.conf_int(alpha=0.20))
pred = pd.DataFrame({"aktual": test.values, "prediksi": np.exp(mu),
                     "bawah80": np.exp(ci[:, 0]), "atas80": np.exp(ci[:, 1])}, index=test.index)
pred.index.name = "bulan"
pred.to_csv("sarima_prediksi_uji.csv")

e = pred["aktual"] - pred["prediksi"]
print(f"MAE  : {e.abs().mean():.2f} miliar")
print(f"RMSE : {np.sqrt((e ** 2).mean()):.2f} miliar")
print(f"MAPE : {(e.abs() / pred['aktual']).mean() * 100:.1f}%")
dalam = ((pred["aktual"] >= pred["bawah80"]) & (pred["aktual"] <= pred["atas80"])).sum()
print(f"Cakupan interval 80%: {dalam} dari {len(pred)} bulan aktual berada di dalam interval")
pred.round(1)

In [ ]:
fig, ax = plt.subplots()
train[-24:].plot(ax=ax, label="Latih (24 bln terakhir)", color="tab:blue")
pred["aktual"].plot(ax=ax, label="Aktual (uji)", color="k", marker="o")
pred["prediksi"].plot(ax=ax, label="Ramalan SARIMA", color="tab:orange", marker="s")
ax.fill_between(pred.index, pred["bawah80"], pred["atas80"], color="tab:orange", alpha=.2, label="Interval 80%")
ax.set_title("Ramalan SARIMA vs aktual pada data uji (miliar Rp)"); ax.legend(); ax.grid(alpha=.3); plt.show()

## 3.7 Menyimpan spesifikasi model

In [ ]:
spesifikasi = {"order": list(ORDER), "seasonal_order": list(SEASONAL), "trend": TREND, "transformasi": "log",
               "AIC": round(float(res.aic), 3), "AICc": round(float(aicc(res, len(train) - BURN)), 3),
               "BIC": round(float(res.bic), 3), "LB_p12": round(float(terpilih["LB_p12"]), 4)}
json.dump(spesifikasi, open("sarima_order.json", "w"), indent=2)
print(spesifikasi)
unduh("sarima_kandidat.csv", "sarima_order.json", "sarima_prediksi_uji.csv")

## Ringkasan Progres 3
- Model SARIMA dipilih lewat grid search 32 kandidat pada log data latih (Jan 2021 – Agu 2025), dengan d = 0 dan D = 1 sesuai Progres 2.
- Spesifikasi terpilih dan hasil diagnostik residual tercetak pada sel 3.3–3.4 dan tersimpan di `sarima_order.json`.
- Ramalan 12 bulan data uji (Sep 2025 – Agu 2026) disimpan di `sarima_prediksi_uji.csv` untuk dibandingkan dengan Random Forest di Progres 5.
- Model **belum** memakai data uji sama sekali dalam pemilihan orde maupun estimasi parameter.